# IBKR Prediction Market / Event Contract Probe

这个 notebook 用来测试 IBKR ForecastTrader / ForecastEx 事件合约框架：

1. 先 dry-run 检查扫描模板是否正常生成。
2. 把 `DRY_RUN = False` 后连接 TWS / IB Gateway 扫描事件合约。
3. 用扫描结果短暂订阅行情，观察 bid/ask、盘口量、spread、volume 和简单流动性评分。

默认不会连接 IBKR；确认 TWS/Gateway 已登录、API enabled、端口正确后再关闭 dry-run。

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
WORKSPACE = cwd.parent if cwd.name == "prediction_market" else cwd
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

import prediction_market.config as pm_config
import prediction_market.ibkr as pm_ibkr
import prediction_market.events as pm_events
import prediction_market.quotes as pm_quotes
import prediction_market.scanner as pm_scanner

pm_config = importlib.reload(pm_config)
pm_ibkr = importlib.reload(pm_ibkr)
pm_events = importlib.reload(pm_events)
pm_quotes = importlib.reload(pm_quotes)
pm_scanner = importlib.reload(pm_scanner)

IBConnectionSettings = pm_config.IBConnectionSettings
ScanSettings = pm_config.ScanSettings
attach_error_collector = pm_ibkr.attach_error_collector
connect_ib = pm_ibkr.connect_ib
detach_error_collector = pm_ibkr.detach_error_collector
contracts_for_event = pm_events.contracts_for_event
event_contract_chain = pm_events.event_contract_chain
fetch_historical_frame = pm_events.fetch_historical_frame
load_contract_frame = pm_events.load_contract_frame
summarize_events = pm_events.summarize_events
fetch_quote_frame = pm_quotes.fetch_quote_frame
load_contracts = pm_quotes.load_contracts
contract_templates = pm_scanner.contract_templates
scan_event_contracts = pm_scanner.scan_event_contracts

DATA_DIR = WORKSPACE / "data"
DATA_DIR.mkdir(exist_ok=True)

print("workspace:", WORKSPACE)
print("data dir:", DATA_DIR)

## 参数

- `DRY_RUN = True`：只检查模板，不连接 IBKR。
- `MARKET_DATA_TYPE = 1`：实时行情；如果只想测延迟行情可用 `3`。
- 如果 TWS 里能看到具体事件合约，把它们的 `localSymbol` 填到 `LOCAL_SYMBOLS`，通常比广泛关键词扫描更稳。

In [ ]:
DRY_RUN = False

HOST = "127.0.0.1"
PORT = 4001          # 4002 常见于 IB Gateway Paper；TWS paper 常见 7497
CLIENT_ID = 602
MARKET_DATA_TYPE = 3 # 1=live, 2=frozen, 3=delayed, 4=delayed frozen

SYMBOL_SEEDS = ["GCE"]  # IBKR 官方 ForecastEx 示例里的 product symbol
EXPIRATIONS = []          # 例：官方示例 20251212
STRIKES = []              # 例：官方示例 6395
RIGHTS = []               # 例：官方示例 C

LOCAL_SYMBOLS = []  # 例：从 TWS 复制来的精确 localSymbol，填这里
SEC_TYPES = ["OPT"]      # ForecastEx event contracts 在 TWS API 里按 OPT 建模
EXCHANGES = ["FORECASTX"] # 注意不是 FORECASTEX

SELECTED_EVENT_SYMBOL = "GCE"
SELECTED_EXPIRY = "20251231"   # None 表示该事件全部到期日
SELECTED_STRIKES = []           # 例：[35500, 36000]；空列表表示不过滤 strike
SELECTED_CHOICES = []           # 例：["YES"] 或 ["YES", "NO"]；空列表表示不过滤

QUOTE_CONTRACT_LIMIT = 20
QUOTE_BATCH_SIZE = 10
QUOTE_WAIT_SECONDS = 20.0

HISTORY_CONTRACT_LIMIT = 4
HISTORY_DURATION = "1 M"
HISTORY_BAR_SIZE = "1 day"
HISTORY_WHAT_TO_SHOW = "TRADES"

CONTRACTS_CSV = DATA_DIR / "prediction_market_contracts.csv"
QUOTES_CSV = DATA_DIR / "prediction_market_quotes.csv"
HISTORY_CSV = DATA_DIR / "prediction_market_history.csv"
ERRORS_CSV = DATA_DIR / "prediction_market_ib_errors.csv"

## 1. Dry-run：检查会向 IBKR 探测哪些合约模板

In [ ]:
scan_settings = ScanSettings(
    symbols=tuple(SYMBOL_SEEDS),
    local_symbols=tuple(LOCAL_SYMBOLS),
    expirations=tuple(EXPIRATIONS),
    strikes=tuple(float(item) for item in STRIKES),
    rights=tuple(RIGHTS),
    sec_types=tuple(SEC_TYPES),
    exchanges=tuple(EXCHANGES),
    keep_all_matches=False,
)

templates = list(contract_templates(scan_settings))
print(f"template count: {len(templates)}")
display(pd.DataFrame([
    {
        "secType": item.secType,
        "symbol": item.symbol,
        "localSymbol": item.localSymbol,
        "expiry": item.lastTradeDateOrContractMonth,
        "strike": item.strike,
        "right": item.right,
        "exchange": item.exchange,
        "currency": item.currency,
    }
    for item in templates[:20]
]))

## 2. 扫描事件合约

第一次建议保持 `MARKET_DATA_TYPE = 3` 和较少 `SYMBOL_SEEDS`。如果空结果，先从 TWS 手动复制几个事件合约 `localSymbol` 到 `LOCAL_SYMBOLS` 再试。

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True：跳过 IBKR 连接。把参数区的 DRY_RUN 改成 False 后运行本 cell。")
else:
    ib_settings = IBConnectionSettings(
        host=HOST,
        port=PORT,
        client_id=CLIENT_ID,
        market_data_type=MARKET_DATA_TYPE,
    )
    ib = connect_ib(ib_settings)
    errors, handler = attach_error_collector(ib)
    try:
        contracts_df = scan_event_contracts(ib, scan_settings)
        contracts_df.to_csv(CONTRACTS_CSV, index=False, encoding="utf-8-sig")
        print(f"wrote {len(contracts_df)} rows -> {CONTRACTS_CSV}")
        events_df = summarize_events(contracts_df)
        print("tradable event summary")
        display(events_df)
        print("raw contract sample")
        display(contracts_df.head(50))
        if errors:
            errors_df = pd.DataFrame(errors)
            errors_df.to_csv(ERRORS_CSV, index=False, encoding="utf-8-sig")
            print(f"wrote {len(errors_df)} IB messages -> {ERRORS_CSV}")
            display(errors_df.tail(20))
    finally:
        detach_error_collector(ib, handler)
        if ib.isConnected():
            ib.disconnect()

## 3. 查看选中事件的合约链

这里按 `expiry / strike / choice(YES/NO)` 展开选中事件的全部相关合约。

In [ ]:
if not CONTRACTS_CSV.exists():
    print(f"找不到扫描结果：{CONTRACTS_CSV}")
else:
    contracts_df = load_contract_frame(str(CONTRACTS_CSV))
    events_df = summarize_events(contracts_df)
    display(events_df)
    chain_df = event_contract_chain(
        contracts_df,
        SELECTED_EVENT_SYMBOL,
        expiry=SELECTED_EXPIRY or None,
    )
    display(chain_df.head(100))

## 4. 按选中事件看报价和流动性

`top_of_book` 表示拿到 bid/ask；`reference_only` 表示只有 close/mark 等参考价，没有当前可交易盘口。

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True：跳过行情订阅。")
elif not CONTRACTS_CSV.exists():
    print(f"找不到扫描结果：{CONTRACTS_CSV}")
else:
    contracts_df = load_contract_frame(str(CONTRACTS_CSV))
    quote_contracts = contracts_for_event(
        contracts_df,
        SELECTED_EVENT_SYMBOL,
        expiry=SELECTED_EXPIRY or None,
        strikes=SELECTED_STRIKES or None,
        choices=SELECTED_CHOICES or None,
    )
    quote_contracts = quote_contracts[:QUOTE_CONTRACT_LIMIT] if QUOTE_CONTRACT_LIMIT else quote_contracts
    print(f"quote event: {SELECTED_EVENT_SYMBOL}")
    print(f"quote contracts: {len(quote_contracts)}")
    if not quote_contracts:
        print("没有可订阅合约。先检查扫描结果或填 LOCAL_SYMBOLS。")
    else:
        ib_settings = IBConnectionSettings(
            host=HOST,
            port=PORT,
            client_id=CLIENT_ID + 1,
            market_data_type=MARKET_DATA_TYPE,
        )
        ib = connect_ib(ib_settings)
        errors, handler = attach_error_collector(ib)
        try:
            quotes_df = fetch_quote_frame(
                ib,
                quote_contracts,
                wait_seconds=QUOTE_WAIT_SECONDS,
                batch_size=QUOTE_BATCH_SIZE,
            )
            quotes_df.to_csv(QUOTES_CSV, index=False, encoding="utf-8-sig")
            print(f"wrote {len(quotes_df)} rows -> {QUOTES_CSV}")
            quote_view_cols = [
                "localSymbol", "bid", "ask", "mid", "last", "markPrice", "close",
                "bidSize", "askSize", "volume", "hasTopOfBook", "hasReferenceData", "marketDataType", "quoteStatus",
            ]
            display(quotes_df[[col for col in quote_view_cols if col in quotes_df.columns]].head(50))
            if errors:
                errors_df = pd.DataFrame(errors)
                errors_df.to_csv(ERRORS_CSV, index=False, encoding="utf-8-sig")
                print(f"wrote {len(errors_df)} IB messages -> {ERRORS_CSV}")
                display(errors_df.tail(20))
        finally:
            detach_error_collector(ib, handler)
            if ib.isConnected():
                ib.disconnect()

## 5. 按选中事件拉历史数据

优先用少量合约测试。若 `TRADES` 返回空，可以把 `HISTORY_WHAT_TO_SHOW` 改成 `BID_ASK` 或 `MIDPOINT` 再试。

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True：跳过历史数据请求。")
elif not CONTRACTS_CSV.exists():
    print(f"找不到扫描结果：{CONTRACTS_CSV}")
else:
    contracts_df = load_contract_frame(str(CONTRACTS_CSV))
    history_contracts = contracts_for_event(
        contracts_df,
        SELECTED_EVENT_SYMBOL,
        expiry=SELECTED_EXPIRY or None,
        strikes=SELECTED_STRIKES or None,
        choices=SELECTED_CHOICES or None,
    )
    history_contracts = history_contracts[:HISTORY_CONTRACT_LIMIT] if HISTORY_CONTRACT_LIMIT else history_contracts
    print(f"history contracts: {len(history_contracts)}")
    if not history_contracts:
        print("没有可请求历史数据的合约。")
    else:
        ib_settings = IBConnectionSettings(
            host=HOST,
            port=PORT,
            client_id=CLIENT_ID + 2,
            market_data_type=MARKET_DATA_TYPE,
        )
        ib = connect_ib(ib_settings)
        errors, handler = attach_error_collector(ib)
        try:
            history_df = fetch_historical_frame(
                ib,
                history_contracts,
                duration=HISTORY_DURATION,
                bar_size=HISTORY_BAR_SIZE,
                what_to_show=HISTORY_WHAT_TO_SHOW,
            )
            history_df.to_csv(HISTORY_CSV, index=False, encoding="utf-8-sig")
            print(f"wrote {len(history_df)} rows -> {HISTORY_CSV}")
            display(history_df.head(100))
            if errors:
                errors_df = pd.DataFrame(errors)
                errors_df.to_csv(ERRORS_CSV, index=False, encoding="utf-8-sig")
                print(f"wrote {len(errors_df)} IB messages -> {ERRORS_CSV}")
                display(errors_df.tail(20))
        finally:
            detach_error_collector(ib, handler)
            if ib.isConnected():
                ib.disconnect()

## 6. 快速查看历史输出

In [ ]:
for path in [CONTRACTS_CSV, QUOTES_CSV, HISTORY_CSV, ERRORS_CSV]:
    print(path, "exists=" + str(path.exists()))
    if path.exists():
        df = pd.read_csv(path)
        print("rows:", len(df))
        display(df.head(10))